In [1]:
import requests
from bs4 import BeautifulSoup

def get_yes24_bestsellers():
    url = "https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001"
    
    # 웹사이트 차단을 방지하기 위한 User-Agent 설정
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # 접속 성공 여부 확인
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 베스트셀러 리스트 아이템 선택 (li 태그)
        # 2024년 개편된 YES24 구조 기준
        book_list = soup.select('#yesBestList > li')
        
        print(f"--- YES24 종합 베스트셀러 목록 ---")
        
        for idx, book in enumerate(book_list, 1):
            # 도서명
            title = book.select_one('.gd_name').get_text(strip=True) if book.select_one('.gd_name') else "제목 없음"
            
            # 저자 정보
            author = book.select_one('.info_auth').get_text(strip=True) if book.select_one('.info_auth') else "저자 미상"
            
            # 가격 (판매가)
            price = book.select_one('.yes_b').get_text(strip=True) if book.select_one('.yes_b') else "가격 정보 없음"
            
            print(f"[{idx}위] {title}")
            print(f"    - 저자: {author}")
            print(f"    - 가격: {price}원")
            print("-" * 30)
            
    except Exception as e:
        print(f"에러가 발생했습니다: {e}")

if __name__ == "__main__":
    get_yes24_bestsellers()

--- YES24 종합 베스트셀러 목록 ---
[1위] 나의 완벽한 장례식
    - 저자: 조현선저
    - 가격: 17,100원
------------------------------
[2위] 진보를 위한 주식투자
    - 저자: 이광수저
    - 가격: 19,800원
------------------------------
[3위] 박곰희 연금 부자 수업
    - 저자: 박곰희저
    - 가격: 18,900원
------------------------------
[4위] 괴테는 모든 것을 말했다
    - 저자: 스즈키 유이저/이지수역
    - 가격: 15,300원
------------------------------
[5위] ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC
    - 저자: ETS저
    - 가격: 19,800원
------------------------------
[6위] ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC
    - 저자: ETS저
    - 가격: 19,800원
------------------------------
[7위] 죽은 왕녀를 위한 파반느 (양장 특별판)
    - 저자: 박민규저
    - 가격: 18,000원
------------------------------
[8위] 자몽살구클럽
    - 저자: 한로로저
    - 가격: 10,800원
------------------------------
[9위] 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상
    - 저자: 최태성저
    - 가격: 17,100원
------------------------------
[10위] 너를 아끼며 살아라
    - 저자: 나태주저
    - 가격: 16,650원
------------------------------
[11위] 모순
    - 저자: 양귀자저
    - 가격: 11,700원
------------------------------
[1

In [2]:
import requests
from bs4 import BeautifulSoup
import csv # 파일 저장을 위해 필요한 라이브러리

def get_yes24_bestsellers():
    url = "https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        book_list = soup.select('#yesBestList > li')
        
        # 데이터를 담을 파이썬 리스트 초기화
        bestseller_data = []
        
        for idx, book in enumerate(book_list, 1):
            title = book.select_one('.gd_name').get_text(strip=True) if book.select_one('.gd_name') else "제목 없음"
            author = book.select_one('.info_auth').get_text(strip=True) if book.select_one('.info_auth') else "저자 미상"
            price = book.select_one('.yes_b').get_text(strip=True) if book.select_one('.yes_b') else "0"
            
            # 각 도서 정보를 딕셔너리로 만들어 리스트에 추가(append)
            book_info = {
                "순위": idx,
                "제목": title,
                "저자": author,
                "가격": price
            }
            bestseller_data.append(book_info)
            
        return bestseller_data
            
    except Exception as e:
        print(f"에러 발생: {e}")
        return []

def save_as_csv(data_list, filename="yes24_bestsellers.csv"):
    """수집된 리스트 데이터를 CSV 파일로 저장하는 함수"""
    if not data_list:
        print("저장할 데이터가 없습니다.")
        return

    # CSV 파일 쓰기 (utf-8-sig는 엑셀에서 한글 깨짐 방지용)
    keys = data_list[0].keys()
    with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader() # 첫 줄에 컬럼명 쓰기
        dict_writer.writerows(data_list) # 리스트 내용 쓰기
        
    print(f"\n파일 저장 완료: {filename}")

if __name__ == "__main__":
    # 1. 크롤링 수행 및 리스트에 저장
    results = get_yes24_bestsellers()
    
    # 2. 결과 확인 (리스트 출력)
    for item in results:
        print(f"[{item['순위']}위] {item['제목']} | {item['저자']} | {item['가격']}원")
        
    # 3. CSV 파일로 저장
    save_as_csv(results)

[1위] 나의 완벽한 장례식 | 조현선저 | 17,100원
[2위] 진보를 위한 주식투자 | 이광수저 | 19,800원
[3위] 박곰희 연금 부자 수업 | 박곰희저 | 18,900원
[4위] 괴테는 모든 것을 말했다 | 스즈키 유이저/이지수역 | 15,300원
[5위] ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC | ETS저 | 19,800원
[6위] ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC | ETS저 | 19,800원
[7위] 죽은 왕녀를 위한 파반느 (양장 특별판) | 박민규저 | 18,000원
[8위] 자몽살구클럽 | 한로로저 | 10,800원
[9위] 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상 | 최태성저 | 17,100원
[10위] 너를 아끼며 살아라 | 나태주저 | 16,650원
[11위] 모순 | 양귀자저 | 11,700원
[12위] 돈의 방정식 | 모건 하우절저/박영준역 | 25,200원
[13위] 자본주의 시대에서 살아남기 위한 최소한의 경제 공부 | 백억남(김욱현)저 | 25,200원
[14위] 1,000만 원으로 3년 안에 300만 원 월배당 만들기 | 인생업(임승현)저 | 20,700원
[15위] 위버멘쉬 | 프리드리히 니체저 | 16,020원
[16위] 엄마가 유령이 되었어! | 노부미글그림/이기웅역 | 10,800원
[17위] 안녕이라 그랬어 | 김애란저 | 15,120원
[18위] 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하 | 최태성저 | 16,650원
[19위] 최소한의 삼국지 | 최태성저/이성원감수 | 17,550원
[20위] 싯다르타 | 헤르만 헤세저/박병덕역 | 7,200원
[21위] 2026 심우철 실전 동형 모의고사 Season 2 | 심우철저 | 13,500원
[22위] 사이토 히토리의 어떻게 살 것인가 | 사이토 히토리저/황미숙역 | 11,700원
[23위] 돈의 심리학 (50만 부 기념 뉴 에디션) | 모건 하우절저/

In [3]:
import requests
from bs4 import BeautifulSoup
import csv
import time # 페이지 이동 사이의 간격을 두기 위해 필요

def get_yes24_bestsellers(start_page, end_page):
    all_bestsellers = []
    
    # 순위를 1위부터 끝까지 매기기 위한 변수
    rank_counter = 1
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    for page in range(start_page, end_page + 1):
        print(f"현재 {page}페이지 수집 중...")
        url = f"https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001&pageNumber={page}"
        
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 도서 목록 선택
            book_list = soup.select('#yesBestList > li')
            
            for book in book_list:
                title = book.select_one('.gd_name').get_text(strip=True) if book.select_one('.gd_name') else "제목 없음"
                author = book.select_one('.info_auth').get_text(strip=True) if book.select_one('.info_auth') else "저자 미상"
                price = book.select_one('.yes_b').get_text(strip=True) if book.select_one('.yes_b') else "0"
                
                all_bestsellers.append({
                    "순위": rank_counter,
                    "제목": title,
                    "저자": author,
                    "가격": price
                })
                rank_counter += 1
            
            # 서버 부하 방지를 위해 페이지당 0.5초 정도 쉽니다.
            time.sleep(0.5)
            
        except Exception as e:
            print(f"{page}페이지 수집 중 에러 발생: {e}")
            break
            
    return all_bestsellers

def save_as_csv(data_list, filename="yes24_bestsellers_1_6.csv"):
    if not data_list:
        return

    keys = data_list[0].keys()
    with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(data_list)
        
    print(f"\n총 {len(data_list)}개의 도서 정보 저장 완료: {filename}")

if __name__ == "__main__":
    # 1. 1페이지부터 6페이지까지 수집
    total_results = get_yes24_bestsellers(1, 6)
    
    # 2. 결과 출력 (상위 5개만 확인)
    print("\n--- 수집 결과 샘플 ---")
    for item in total_results[:5]:
        print(f"[{item['순위']}위] {item['제목']}")
        
    # 3. 파일 저장
    save_as_csv(total_results)

현재 1페이지 수집 중...
현재 2페이지 수집 중...
현재 3페이지 수집 중...
현재 4페이지 수집 중...
현재 5페이지 수집 중...
현재 6페이지 수집 중...

--- 수집 결과 샘플 ---
[1위] 나의 완벽한 장례식
[2위] 진보를 위한 주식투자
[3위] 박곰희 연금 부자 수업
[4위] 괴테는 모든 것을 말했다
[5위] ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC

총 144개의 도서 정보 저장 완료: yes24_bestsellers_1_6.csv


In [4]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time

def get_yes24_data(start_page, end_page):
    """1~6페이지 데이터를 수집하여 리스트로 반환하는 함수"""
    all_books = []
    rank_counter = 1
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    for page in range(start_page, end_page + 1):
        print(f"현재 {page}페이지 수집 중...")
        url = f"https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001&pageNumber={page}"
        
        try:
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.text, 'html.parser')
            book_list = soup.select('#yesBestList > li')
            
            for book in book_list:
                title = book.select_one('.gd_name').get_text(strip=True) if book.select_one('.gd_name') else "제목 없음"
                author = book.select_one('.info_auth').get_text(strip=True) if book.select_one('.info_auth') else "저자 미상"
                price = book.select_one('.yes_b').get_text(strip=True).replace(',', '') if book.select_one('.yes_b') else "0"
                
                # DB 입력을 위해 튜플(Tuple) 형태로 저장
                all_books.append((rank_counter, title, author, int(price)))
                rank_counter += 1
            
            time.sleep(0.5)
        except Exception as e:
            print(f"오류 발생: {e}")
    return all_books

def save_to_sqlite(book_data):
    """수집된 리스트를 SQLite DB에 저장하는 함수"""
    # 1. DB 연결 (파일이 없으면 새로 생성됨)
    conn = sqlite3.connect('yes24_bestsellers.db')
    cur = conn.cursor()

    # 2. 테이블 생성 (이미 있다면 삭제 후 새로 생성하거나 유지 선택 가능)
    cur.execute("DROP TABLE IF EXISTS bestsellers")
    cur.execute("""
        CREATE TABLE bestsellers (
            rank INTEGER,
            title TEXT,
            author TEXT,
            price INTEGER
        )
    """)

    # 3. 데이터 대량 삽입 (executemany 사용)
    cur.executemany("INSERT INTO bestsellers VALUES (?, ?, ?, ?)", book_data)

    # 4. 변경사항 저장 및 종료
    conn.commit()
    print(f"\nDB 저장 완료! 총 {len(book_data)}개의 행이 입력되었습니다.")
    
    # 데이터 확인용 쿼리 (상위 10개만 출력)
    print("\n--- DB 저장 데이터 확인 (상위 10개) ---")
    cur.execute("SELECT * FROM bestsellers LIMIT 10")
    for row in cur.fetchall():
        print(row)
        
    conn.close()

if __name__ == "__main__":
    # 데이터 수집 (1~6페이지)
    data = get_yes24_data(1, 6)
    
    # DB 저장 실행
    if data:
        save_to_sqlite(data)

현재 1페이지 수집 중...
현재 2페이지 수집 중...
현재 3페이지 수집 중...
현재 4페이지 수집 중...
현재 5페이지 수집 중...
현재 6페이지 수집 중...

DB 저장 완료! 총 144개의 행이 입력되었습니다.

--- DB 저장 데이터 확인 (상위 10개) ---
(1, '나의 완벽한 장례식', '조현선저', 17100)
(2, '진보를 위한 주식투자', '이광수저', 19800)
(3, '박곰희 연금 부자 수업', '박곰희저', 18900)
(4, '괴테는 모든 것을 말했다', '스즈키 유이저/이지수역', 15300)
(5, 'ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC', 'ETS저', 19800)
(6, 'ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC', 'ETS저', 19800)
(7, '죽은 왕녀를 위한 파반느 (양장 특별판)', '박민규저', 18000)
(8, '자몽살구클럽', '한로로저', 10800)
(9, '2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상', '최태성저', 17100)
(10, '너를 아끼며 살아라', '나태주저', 16650)
